In [0]:
customers_df = spark.table(
    "ecommerce_project.project.silver_customers"
)

products_df = spark.table(
    "ecommerce_project.project.silver_products"
)

orders_df = spark.table(
    "ecommerce_project.project.silver_orders"
)

payments_df = spark.table(
    "ecommerce_project.project.silver_payments"
)

deliveries_df = spark.table(
    "ecommerce_project.project.silver_deliveries"
)

In [0]:
from pyspark.sql import functions as F
payment_cardinality = (
    payments_df
    .groupBy("order_id")
    .count()
    .orderBy(F.desc("count"))
)

display(payment_cardinality)

In [0]:
delivery_cardinality = (
    deliveries_df
    .groupBy("order_id")
    .count()
    .orderBy(F.desc("count"))
)

display(delivery_cardinality)

In [0]:
payment_summary_df = (
    payments_df
    .groupBy("order_id")
    .agg(
        F.sum("amount").alias("total_payment_amount"),
        F.count("payment_id").alias("payment_count"),
        F.max("payment_date").alias("latest_payment_date")
    )
)

In [0]:
delivery_summary_df = (
    deliveries_df
    .groupBy("order_id")
    .agg(
        F.max("delivery_date").alias("latest_delivery_date"),
        F.max("delivery_status").alias("delivery_status"),
        F.max("delivery_city").alias("delivery_city"),
        F.count("delivery_id").alias("delivery_count")
    )
)

In [0]:
gold_order_details_df = (
    orders_df.alias("o")
    .join(
        customers_df.alias("c"),
        F.col("o.customer_id") == F.col("c.customer_id"),
        "left"
    )
    .join(
        products_df.alias("p"),
        F.col("o.product_id") == F.col("p.product_id"),
        "left"
    )
    .join(
        payment_summary_df.alias("pay"),
        F.col("o.order_id") == F.col("pay.order_id"),
        "left"
    )
    .join(
        delivery_summary_df.alias("d"),
        F.col("o.order_id") == F.col("d.order_id"),
        "left"
    )
    .select(
        F.col("o.order_id"),
        F.col("o.order_date"),

        F.col("o.customer_id"),
        F.col("c.customer_name"),
        F.col("c.email"),
        F.col("c.city").alias("customer_city"),
        F.col("c.state"),

        F.col("o.product_id"),
        F.col("p.product_name"),
        F.col("p.category"),
        F.col("p.unit_price"),

        F.col("o.quantity"),
        F.col("o.order_status"),

        F.col("pay.total_payment_amount"),
        F.col("pay.payment_count"),
        F.col("pay.latest_payment_date"),

        F.col("d.latest_delivery_date"),
        F.col("d.delivery_status"),
        F.col("d.delivery_city"),
        F.col("d.delivery_count")
    )
)

In [0]:
gold_order_details_df = (
    gold_order_details_df
    .withColumn(
        "order_revenue",
        F.col("quantity") * F.col("unit_price")
    )
)

In [0]:
gold_order_details_df \
    .groupBy("order_id") \
    .count() \
    .filter(F.col("count") > 1) \
    .show()

In [0]:
gold_order_details_df.select(
    F.count("*").alias("total_orders"),
    F.sum(F.col("customer_name").isNull().cast("int")).alias("missing_customer"),
    F.sum(F.col("product_name").isNull().cast("int")).alias("missing_product"),
    F.sum(F.col("total_payment_amount").isNull().cast("int")).alias("missing_payment"),
    F.sum(F.col("delivery_status").isNull().cast("int")).alias("missing_delivery")
).show()

In [0]:
gold_order_details_df = (
    gold_order_details_df
    .withColumn(
        "payment_available",
        F.when(F.col("payment_count").isNull(), F.lit(False))
         .otherwise(F.lit(True))
    )
    .withColumn(
        "delivery_available",
        F.when(F.col("delivery_count").isNull(), F.lit(False))
         .otherwise(F.lit(True))
    )
)



In [0]:
print("Silver Orders:", orders_df.count())
print("Gold Orders:", gold_order_details_df.count())

print(
    "Duplicate Orders:",
    gold_order_details_df
        .groupBy("order_id")
        .count()
        .filter(F.col("count") > 1)
        .count()
)

In [0]:
gold_order_details_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "ecommerce_project.project.gold_order_details"
    )

In [0]:
gold_order_details = spark.table(
    "ecommerce_project.project.gold_order_details"
)

print("Gold Order Details:", gold_order_details.count())

display(
    gold_order_details.limit(20)
)

In [0]:
customer_summary_df = (
    customers_df.alias("c")
    .join(
        gold_order_details_df.alias("o"),
        F.col("c.customer_id") == F.col("o.customer_id"),
        "left"
    )
    .groupBy(
        F.col("c.customer_id"),
        F.col("c.customer_name"),
        F.col("c.email"),
        F.col("c.city"),
        F.col("c.state")
    )
    .agg(
        F.countDistinct("o.order_id").alias("total_orders"),
        F.sum("o.quantity").alias("total_quantity"),
        F.sum("o.order_revenue").alias("total_spend"),
        F.sum(
            F.when(
                F.col("o.order_status") == "delivered",
                1
            ).otherwise(0)
        ).alias("delivered_orders"),
        F.sum(
            F.when(
                F.col("o.order_status") == "cancelled",
                1
            ).otherwise(0)
        ).alias("cancelled_orders"),
        F.sum(
            F.when(
                F.col("o.payment_available") == True,
                1
            ).otherwise(0)
        ).alias("orders_with_payment")
    )
)

In [0]:
customer_summary_df = (
    customer_summary_df
    .fillna({
        "total_orders": 0,
        "total_quantity": 0,
        "total_spend": 0,
        "delivered_orders": 0,
        "cancelled_orders": 0,
        "orders_with_payment": 0
    })
)

In [0]:
product_summary_df = (
    products_df.alias("p")
    .join(
        gold_order_details_df.alias("o"),
        F.col("p.product_id") == F.col("o.product_id"),
        "left"
    )
    .groupBy(
        F.col("p.product_id"),
        F.col("p.product_name"),
        F.col("p.category"),
        F.col("p.unit_price")
    )
    .agg(
        F.countDistinct("o.order_id").alias("total_orders"),
        F.sum("o.quantity").alias("total_quantity_sold"),
        F.sum("o.order_revenue").alias("total_revenue")
    )
)

In [0]:
product_summary_df = (
    product_summary_df
    .fillna({
        "total_orders": 0,
        "total_quantity_sold": 0,
        "total_revenue": 0
    })
)

In [0]:
product_summary_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "ecommerce_project.project.gold_product_summary"
    )

In [0]:
customer_summary_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "ecommerce_project.project.gold_customer_summary"
    )

In [0]:
gold_order_details = spark.table(
    "ecommerce_project.project.gold_order_details"
)

gold_customer_summary = spark.table(
    "ecommerce_project.project.gold_customer_summary"
)

gold_product_summary = spark.table(
    "ecommerce_project.project.gold_product_summary"
)

print("Gold Order Details:", gold_order_details.count())
print("Gold Customer Summary:", gold_customer_summary.count())
print("Gold Product Summary:", gold_product_summary.count())

In [0]:
gold_order_details_df \
    .groupBy("order_id") \
    .count() \
    .filter(F.col("count") > 1) \
    .show()

print("Total Gold Orders:", gold_order_details_df.select("order_id").distinct().count())

In [0]:
customer_summary_df \
    .groupBy("customer_id") \
    .count() \
    .filter(F.col("count") > 1) \
    .show()

print("Total Gold Customers:", customer_summary_df.select("customer_id").distinct().count())

In [0]:
product_summary_df \
    .groupBy("product_id") \
    .count() \
    .filter(F.col("count") > 1) \
    .show()

print("Total Gold Products:", product_summary_df.select("product_id").distinct().count())

In [0]:

expected_revenue_df = (
    orders_df.alias("o")
    .join(
        products_df.alias("p"),
        F.col("o.product_id") == F.col("p.product_id"),
        "inner"
    )
    .select(
        F.sum(
            F.col("o.quantity") * F.col("p.unit_price")
        ).alias("expected_revenue")
    )
)

gold_revenue_df = (
    gold_order_details_df
    .select(
        F.sum("order_revenue").alias("gold_revenue")
    )
)

expected_revenue_df.show()
gold_revenue_df.show()

In [0]:
customer_order_check_df = (
    gold_order_details_df
    .groupBy("customer_id")
    .agg(
        F.countDistinct("order_id").alias("expected_orders"),
        F.sum("quantity").alias("expected_quantity"),
        F.sum("order_revenue").alias("expected_spend")
    )
)

customer_order_check_df.show(10)

In [0]:
product_order_check_df = (
    gold_order_details_df
    .groupBy("product_id")
    .agg(
        F.countDistinct("order_id").alias("expected_orders"),
        F.sum("quantity").alias("expected_quantity"),
        F.sum("order_revenue").alias("expected_revenue")
    )
)

product_order_check_df.show(10)

In [0]:
gold_order_details = spark.table(
    "ecommerce_project.project.gold_order_details"
)

gold_customer_summary = spark.table(
    "ecommerce_project.project.gold_customer_summary"
)

gold_product_summary = spark.table(
    "ecommerce_project.project.gold_product_summary"
)

print("Gold Order Details:", gold_order_details.count())
print("Gold Customer Summary:", gold_customer_summary.count())
print("Gold Product Summary:", gold_product_summary.count())

In [0]:
print("========== GOLD VALIDATION ==========")

print("Gold Order Details:", gold_order_details.count())
print("Gold Customer Summary:", gold_customer_summary.count())
print("Gold Product Summary:", gold_product_summary.count())

print("\nDuplicate Order IDs:")
gold_order_details \
    .groupBy("order_id") \
    .count() \
    .filter(F.col("count") > 1) \
    .show()

print("Duplicate Customer IDs:")
gold_customer_summary \
    .groupBy("customer_id") \
    .count() \
    .filter(F.col("count") > 1) \
    .show()

print("Duplicate Product IDs:")
gold_product_summary \
    .groupBy("product_id") \
    .count() \
    .filter(F.col("count") > 1) \
    .show()